In [1]:
# ============================================================
# CELL 1 — INSTALL DEPENDENCIES
# ============================================================
!pip install -q transformers peft accelerate bitsandbytes datasets trl scikit-learn openai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 51.7 MB/s eta 0:00:00


In [ ]:
# Runtime -> restart session for bitsandbytes

In [1]:
# ============================================================
# CELL 2 — MOUNT GOOGLE DRIVE & IMPORTS
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
import os
import shutil
import json
import torch
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig # Simplified TRL imports

DRIVE_ROOT = '/content/drive/MyDrive/298B_WB2'
# Copy the JSONL from drive
shutil.copy(f'{DRIVE_ROOT}/planner_train_graphrag.jsonl', '/content/planner_train_graphrag.jsonl')


Mounted at /content/drive


'/content/planner_train_graphrag.jsonl'

In [2]:
# ============================================================
# CELL 3 — CONFIGURATION & DATA PREP (80-10-10 Split) — Run 4
# ============================================================
BASE_MODEL = 'Qwen/Qwen2.5-Coder-7B-Instruct'
OUTPUT_DIR = f'{DRIVE_ROOT}/training_run_planner_v4'
ADAPTER_DIR = f'{DRIVE_ROOT}/planner_lora_adapter_v4'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(ADAPTER_DIR, exist_ok=True)

records = []
with open('/content/planner_train_graphrag.jsonl', 'r') as f:
    for line in f:
        if line.strip():
            data = json.loads(line)
            system_prompt = (
                'You are AutoBot Planner. Decide if a code change is required and output STRICT '
                'plain text only — no markdown, bold, or prose outside the required fields. '
                'Use exact fields: REQUIRES_CODE_CHANGE, REASON, and PLAN (for YES only). '
                'STRICT GROUNDING: You are an engineering tool. You must only select target files '
                'from the provided context blocks. Do not hallucinate paths or add file extensions '
                'not present in the retrieval. If a change is needed but no file is in context, '
                'identify the most relevant directory from the tree-sitter context. '
                'The user message may include --- RETRIEVAL EVIDENCE ---; treat it as context only. '
                'Do not output a CONFIDENCE field.'
            )

            # SPLIT: The prompt is System + User + the 'Assistant' start tag
            prompt_part = (
                f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
                f"<|im_start|>user\n{data['input']}<|im_end|>\n"
                f"<|im_start|>assistant\n"
            )
            # The completion is only the model's output
            completion_part = f"{data['output']}<|im_end|>"

            records.append({
                'prompt': prompt_part,
                'completion': completion_part,
                'label': 'YES' if 'REQUIRES_CODE_CHANGE: YES' in data['output'] else 'NO',
                'original_output': data['output']
            })

df = pd.DataFrame(records)
train_val_df, test_df = train_test_split(df, test_size=0.1, stratify=df['label'], random_state=42)
train_df, val_df = train_test_split(train_val_df, test_size=0.1111, stratify=train_val_df['label'], random_state=42)

# Convert to Dataset with both prompt and completion columns
train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test (Blind): {len(test_df)}')

Train: 2400 | Val: 300 | Test (Blind): 300


In [3]:
# ============================================================
# CELL 4 — VERIFY TOKENIZER & LOAD MODEL
# ============================================================
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

TOKENIZER = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
TOKENIZER.pad_token = TOKENIZER.eos_token
TOKENIZER.padding_side = 'right'

print(f'Loading {BASE_MODEL}...')
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.bfloat16,
    device_map='auto', trust_remote_code=True
)
model.config.use_cache = False

model = get_peft_model(model, LoraConfig(
    r=32, lora_alpha=128,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj', 'gate_proj', 'up_proj'],
    lora_dropout=0.05,
    bias='none', task_type=TaskType.CAUSAL_LM,
    modules_to_save=['lm_head', 'embed_tokens']
))
model.print_trainable_parameters()


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


trainable params: 1,150,550,016 || all params: 8,766,166,528 || trainable%: 13.1249


In [4]:
# ============================================================
# CELL 5 — TRAIN (Run 4: cosine LR, standard CE / no class weights)
# ============================================================
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=5e-5,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    logging_steps=10,
    max_length=4096, # Context window for Airflow
    num_train_epochs=3,
    eval_strategy='steps',
    eval_steps=50,
    save_strategy='steps',
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    bf16=True,
    report_to='none',
    completion_only_loss=True, # loss only on assistant completion (prompt masked)
    packing=False, # must be False when completion_only_loss=True
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=training_args,
)

trainer.train()
trainer.model.save_pretrained(ADAPTER_DIR)
TOKENIZER.save_pretrained(ADAPTER_DIR)
print('Adapter saved (best eval_loss checkpoint).')

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/2400 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2400 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
50,0.862627,0.813774
100,0.816888,0.751941
150,0.718250,0.724772
200,0.599945,0.721756
250,0.526539,0.709682
300,0.669445,0.706388
350,0.438968,0.726645
400,0.486790,0.727847
450,0.438652,0.727669


Adapter saved (best eval_loss checkpoint).


In [ ]:
# ==== Run this POST-RESTART to load your persisted adapter on drive for CELL 6 evals ====
from peft import PeftModel
# This replaces the 'fresh' LoRA layers with your trained ones
model = PeftModel.from_pretrained(model, ADAPTER_DIR)
print("Successfully loaded trained Planner adapters!")


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Successfully loaded trained Planner adapters!


/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:598: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0

In [5]:
# ============================================================
# CELL 6 — EVAL: COMPLETE PERFORMANCE & GATE AUDIT (Run 4)
# Token-level log-probs for gate calibration are deferred until orchestrator integration.
# Text quality: G-Eval via OpenAI (gpt-4o-mini) + Colab Secret OPENAI_API_KEY — not BERTScore.
# Retrieval: Recall@1 / @3 / @6 on the same held-out test_df rows (full split unless MAX_TEST_EVAL_ROWS).
# Reports: JSON + metrics_readout.txt + CSV (no image export; plot from saved numbers).
# ============================================================
import torch
import re
import os
import json
import time
import shutil
import datetime
from collections import Counter

import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report, f1_score
from tqdm.auto import tqdm

# Persist evals on Drive; wipe v4 on each run so reruns are clean.
EVAL_DIR = os.path.join(DRIVE_ROOT, 'evals', 'planner', 'v4')
if os.path.isdir(EVAL_DIR):
    shutil.rmtree(EVAL_DIR)
os.makedirs(EVAL_DIR, exist_ok=True)
print(f'Eval output directory (reset): {EVAL_DIR}')

# --- 1. EVALUATION HELPERS ---
def extract_files(text):
    clean_text = text.replace('**', '')
    match = re.search(r'(?i)Target files:\s*(.*?)(?:\n\s*[-\*•]*\s*Test strategy:|$)', clean_text, re.DOTALL)
    if not match: return set()
    files_block = match.group(1).strip()
    if files_block.lower() in ['none', 'none.', 'n/a']: return set()
    files = re.findall(r'(?m)^\s*[-\*•]\s+([^\n\r]+)', files_block)
    return set(f.strip() for f in files if f.strip() not in ['None', 'None.'])

def extract_classification(text):
    match = re.search(r'REQUIRES_CODE_CHANGE:\s*(YES|NO)', text, re.IGNORECASE)
    return match.group(1).upper() if match else 'UNKNOWN'

def check_structure_score(text):
    is_yes = "REQUIRES_CODE_CHANGE: YES" in text.upper()
    required = [r'REQUIRES_CODE_CHANGE:', r'REASON:']
    if is_yes:
        required += [r'PLAN:', r'-\s*What to change:', r'-\s*Target files:', r'-\s*Test strategy:']
    found = sum(1 for pattern in required if re.search(pattern, text, re.IGNORECASE))
    return found / len(required)

# --- 2. MULTI-PATH GENERATION ENGINE ---
def generate_k_predictions(full_prompt, use_adapter=True, k=1):
    inputs = TOKENIZER(full_prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        if use_adapter:
            outputs = model.generate(
                **inputs, max_new_tokens=400, do_sample=(k > 1),
                temperature=0.7 if k > 1 else 1.0, top_p=0.9, num_return_sequences=k
            )
        else:
            with model.disable_adapter():
                outputs = model.generate(
                    **inputs, max_new_tokens=400, do_sample=(k > 1),
                    temperature=0.7 if k > 1 else 1.0, top_p=0.9, num_return_sequences=k
                )
    return [TOKENIZER.decode(out[inputs['input_ids'].shape[1]:], skip_special_tokens=True) for out in outputs]

# --- 3. EVALUATION LOOP (held-out test_df only; same 80/10/10 test split as Cell 3) ---
import statistics

# None = evaluate every row in test_df. Set e.g. 100 to cap cost/time (first N rows, deterministic order).
MAX_TEST_EVAL_ROWS = None

eval_samples = test_df.reset_index(drop=True)
if MAX_TEST_EVAL_ROWS is not None:
    eval_samples = eval_samples.iloc[: int(MAX_TEST_EVAL_ROWS)]

metrics = {
    'FT': {
        'recall_1': [], 'recall_3': [], 'recall_6': [], 'structure': [], 'texts': [], 'classes': [],
        'gen_sec': [], 'geval_per_row': [],
    },
    'Base': {
        'recall_1': [], 'recall_3': [], 'recall_6': [], 'structure': [], 'texts': [], 'classes': [],
        'gen_sec': [], 'geval_per_row': [],
    },
}
gt_classes = []

n_test = len(test_df)
n_eval = len(eval_samples)
print(
    f"Eval on held-out test split: {n_eval} row(s) "
    f"(full test_df has {n_test}; cap MAX_TEST_EVAL_ROWS={MAX_TEST_EVAL_ROWS})."
)
print("Comparing Base vs. Fine-Tuned (k=6) on identical test prompts...")
for row in tqdm(eval_samples.itertuples(), total=len(eval_samples)):
    gt_output = row.original_output
    gt_files = extract_files(gt_output)
    gt_class = 'YES' if 'REQUIRES_CODE_CHANGE: YES' in gt_output else 'NO'
    gt_classes.append(gt_class)

    for mode in ['FT', 'Base']:
        t0 = time.perf_counter()
        preds = generate_k_predictions(row.prompt, use_adapter=(mode == 'FT'), k=6)
        gen_sec = time.perf_counter() - t0
        metrics[mode]['gen_sec'].append(gen_sec)
        top1 = preds[0]

        # Clean 'UNKNOWN' for F1 calculation (treat as NO for safety)
        pred_class = extract_classification(top1)
        metrics[mode]['classes'].append('NO' if pred_class == 'UNKNOWN' else pred_class)
        metrics[mode]['texts'].append(top1)
        metrics[mode]['structure'].append(check_structure_score(top1))

        if gt_class == 'YES' and len(gt_files) > 0:
            pred_file_sets = [extract_files(p) for p in preds]
            metrics[mode]['recall_1'].append(len(gt_files.intersection(pred_file_sets[0])) / len(gt_files))
            metrics[mode]['recall_3'].append(len(gt_files.intersection(set().union(*pred_file_sets[:3]))) / len(gt_files))
            metrics[mode]['recall_6'].append(len(gt_files.intersection(set().union(*pred_file_sets[:6]))) / len(gt_files))

print("\n--- Generation latency (wall-clock per k=6 generate call) ---")
for mode in ['FT', 'Base']:
    g = metrics[mode]['gen_sec']
    if not g:
        continue
    print(
        f"{mode}: mean={statistics.mean(g):.4f}s  median={statistics.median(g):.4f}s  "
        f"min={min(g):.4f}s  max={max(g):.4f}s  (n={len(g)} calls, one per test row)"
    )
combined = [metrics['FT']['gen_sec'][i] + metrics['Base']['gen_sec'][i] for i in range(len(metrics['FT']['gen_sec']))]
print(
    f"FT+Base per test row (both models): mean={statistics.mean(combined):.4f}s  "
    f"median={statistics.median(combined):.4f}s"
)

timing_df = pd.DataFrame({
    'test_row': range(len(metrics['FT']['gen_sec'])),
    'ft_gen_s': metrics['FT']['gen_sec'],
    'base_gen_s': metrics['Base']['gen_sec'],
})
timing_df['both_models_s'] = timing_df['ft_gen_s'] + timing_df['base_gen_s']
print("\nPer-test-row latency (s) for one k=6 generation each (same row = same prompt):")
_timing_show = timing_df.round(4)
if len(_timing_show) <= 60:
    print(_timing_show.to_string(index=False))
else:
    print(_timing_show.head(20).to_string(index=False))
    print(f"... ({len(_timing_show) - 40} rows omitted) ...")
    print(_timing_show.tail(20).to_string(index=False))

# --- 4. G-EVAL (OpenAI) + reporting (same test rows for Base vs FT) ---
from openai import OpenAI

try:
    from google.colab import userdata
except ImportError:
    userdata = None

GEVAL_MODEL = "gpt-4o-mini"  # Colab: set OPENAI_API_KEY in Secrets (lock icon)


def _openai_api_key():
    if userdata is None:
        raise RuntimeError("Run on Google Colab and use Secrets for OPENAI_API_KEY.")
    try:
        k = userdata.get("OPENAI_API_KEY")
    except Exception as e:
        raise RuntimeError("Could not read OPENAI_API_KEY from Colab Secrets.") from e
    if not k or not str(k).strip():
        raise RuntimeError("OPENAI_API_KEY secret is missing or empty.")
    return str(k).strip()


def excerpt_current_ticket(full_prompt: str, max_chars: int = 2800) -> str:
    if "--- CURRENT TICKET ---" in full_prompt:
        return full_prompt.split("--- CURRENT TICKET ---", 1)[-1][:max_chars].strip()
    return full_prompt[-max_chars:].strip()


def _parse_geval_json(text: str) -> dict:
    t = text.strip()
    if t.startswith("```"):
        t = re.sub(r"^```(?:json)?\s*", "", t, flags=re.IGNORECASE)
        t = re.sub(r"\s*```$", "", t).strip()
    return json.loads(t)


def geval_scores_one(client, issue_excerpt: str, reference: str, prediction: str) -> float:
    system = (
        "You are a strict evaluator for an engineering planner (G-Eval style). "
        "Each score is an integer 1 (poor) to 5 (excellent). Reply with ONLY a JSON object, no prose."
    )
    user = f"""Score the MODEL OUTPUT against the REFERENCE for this ticket.

Dimensions (each 1-5):
- relevance: Addresses the ticket and the same YES/NO code-change decision as the reference.
- consistency_with_ref: Semantic alignment with the reference answer (not verbatim).
- format_clarity: Correct fields (REQUIRES_CODE_CHANGE, REASON, PLAN only when YES), plain text.

ISSUE (excerpt):
{issue_excerpt}

REFERENCE (gold assistant):
{reference}

MODEL OUTPUT:
{prediction}

Return only JSON: {{"relevance": <int>, "consistency_with_ref": <int>, "format_clarity": <int>}}"""

    resp = client.chat.completions.create(
        model=GEVAL_MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        temperature=0,
        max_tokens=120,
    )
    raw = (resp.choices[0].message.content or "").strip()
    d = _parse_geval_json(raw)
    r = max(1, min(5, int(d.get("relevance", 3))))
    c = max(1, min(5, int(d.get("consistency_with_ref", 3))))
    f = max(1, min(5, int(d.get("format_clarity", 3))))
    return (r + c + f) / 3.0


_geval_client = OpenAI(api_key=_openai_api_key())
_gt_refs = [r.original_output for r in eval_samples.itertuples()]
_gt_prompts = [r.prompt for r in eval_samples.itertuples()]

summary_results = []
for mode in ['Base', 'FT']:
    clf_f1 = f1_score(gt_classes, metrics[mode]['classes'], pos_label='YES')
    geval_means = []
    for pred, ref, full_prompt in tqdm(
        list(zip(metrics[mode]['texts'], _gt_refs, _gt_prompts)),
        desc=f"G-Eval {mode}",
    ):
        ex = excerpt_current_ticket(full_prompt)
        try:
            geval_means.append(geval_scores_one(_geval_client, ex, ref, pred))
        except Exception as e:
            print("G-Eval row failed:", e)
            geval_means.append(float("nan"))
        time.sleep(0.05)

    geval_avg = float(pd.Series(geval_means).mean(skipna=True))
    metrics[mode]['geval_per_row'] = list(geval_means)

    summary_results.append({
        'Model': 'Fine-Tuned (AutoBot Planner)' if mode == 'FT' else 'Base (Qwen 2.5 Coder)',
        'Structure Score': f"{sum(metrics[mode]['structure']) / len(metrics[mode]['structure']):.2%}",
        'Classification F1': f"{clf_f1:.4f}",
        'Recall@1': sum(metrics[mode]['recall_1']) / len(metrics[mode]['recall_1']) if metrics[mode]['recall_1'] else 0,
        'Recall@3': sum(metrics[mode]['recall_3']) / len(metrics[mode]['recall_3']) if metrics[mode]['recall_3'] else 0,
        'Recall@6': sum(metrics[mode]['recall_6']) / len(metrics[mode]['recall_6']) if metrics[mode]['recall_6'] else 0,
        'Gen mean (s)': statistics.mean(metrics[mode]['gen_sec']) if metrics[mode]['gen_sec'] else 0.0,
        'G-Eval avg (1-5)': geval_avg,
    })

print("\n" + "="*80 + "\nFINAL PERFORMANCE COMPARISON\n" + "="*80)
print(pd.DataFrame(summary_results).to_string(index=False))

# --- 5. Detailed JSON/CSV report (no image export; plot from these values locally) ---
def _trunc_txt(s, n=500):
    s = s or ''
    return s if len(s) <= n else s[:n] + '…[truncated]'

_sr_base = summary_results[0]
_sr_ft = summary_results[1]

_rep_base_dict = classification_report(
    gt_classes, metrics['Base']['classes'], labels=['YES', 'NO'], output_dict=True, zero_division=0
)
_rep_ft_dict = classification_report(
    gt_classes, metrics['FT']['classes'], labels=['YES', 'NO'], output_dict=True, zero_division=0
)
_cm_base = confusion_matrix(gt_classes, metrics['Base']['classes'], labels=['YES', 'NO']).tolist()
_cm_ft = confusion_matrix(gt_classes, metrics['FT']['classes'], labels=['YES', 'NO']).tolist()

_recall_by_model = {
    'Base': {
        'Recall@1': float(_sr_base['Recall@1']),
        'Recall@3': float(_sr_base['Recall@3']),
        'Recall@6': float(_sr_base['Recall@6']),
    },
    'FT': {
        'Recall@1': float(_sr_ft['Recall@1']),
        'Recall@3': float(_sr_ft['Recall@3']),
        'Recall@6': float(_sr_ft['Recall@6']),
    },
}
_latency_by_model = {
    'Base': {
        'gen_sec_mean': float(statistics.mean(metrics['Base']['gen_sec'])) if metrics['Base']['gen_sec'] else None,
        'gen_sec_median': float(statistics.median(metrics['Base']['gen_sec'])) if metrics['Base']['gen_sec'] else None,
    },
    'FT': {
        'gen_sec_mean': float(statistics.mean(metrics['FT']['gen_sec'])) if metrics['FT']['gen_sec'] else None,
        'gen_sec_median': float(statistics.median(metrics['FT']['gen_sec'])) if metrics['FT']['gen_sec'] else None,
    },
}

_per_rows = []
for i in range(len(gt_classes)):
    _per_rows.append({
        'test_row': int(i),
        'gt_class': gt_classes[i],
        'pred_base': metrics['Base']['classes'][i],
        'pred_ft': metrics['FT']['classes'][i],
        'structure_base': float(metrics['Base']['structure'][i]),
        'structure_ft': float(metrics['FT']['structure'][i]),
        'geval_base': float(metrics['Base']['geval_per_row'][i]),
        'geval_ft': float(metrics['FT']['geval_per_row'][i]),
        'gen_sec_base': float(metrics['Base']['gen_sec'][i]),
        'gen_sec_ft': float(metrics['FT']['gen_sec'][i]),
        'pred_preview_base': _trunc_txt(metrics['Base']['texts'][i]),
        'pred_preview_ft': _trunc_txt(metrics['FT']['texts'][i]),
        'gold_preview': _trunc_txt(_gt_refs[i]),
    })

_metrics_readout = []
_metrics_readout.append('=== Planner eval (text readout) ===')
_metrics_readout.append(f"eval_dir={EVAL_DIR}")
_metrics_readout.append(f"test rows evaluated: {n_eval} (full test_df has {n_test})")
_metrics_readout.append(f"MAX_TEST_EVAL_ROWS cap: {MAX_TEST_EVAL_ROWS}")
_metrics_readout.append(f"G-Eval model: {GEVAL_MODEL}")
_metrics_readout.append('label counts (ground truth): ' + str(dict(Counter(gt_classes))))
for _m in summary_results:
    _metrics_readout.append('')
    for k, v in _m.items():
        _metrics_readout.append(f"  {k}: {v}")
_metrics_readout.append('')
_metrics_readout.append('Confusion (Base), rows: YES,NO pred x YES,NO gt: ' + str(_cm_base))
_metrics_readout.append('Confusion (FT),  rows: same order: ' + str(_cm_ft))
_metrics_readout.append('Recall (mean, YES+files rows only): ' + str(_recall_by_model))
_metrics_readout.append('Gen latency s (k=6): ' + str(_latency_by_model))
metrics_readout_text = '\n'.join(_metrics_readout)

_eval_meta = {
    'created_utc': datetime.datetime.utcnow().replace(microsecond=0).isoformat() + 'Z',
    'eval_dir': EVAL_DIR,
    'n_test_total': int(n_test),
    'n_eval_rows': int(n_eval),
    'max_test_eval_rows': MAX_TEST_EVAL_ROWS,
    'geval_model': GEVAL_MODEL,
    'test_label_counts': dict(Counter(gt_classes)),
    'latency_seconds_summary': _latency_by_model,
    'recall_by_model': _recall_by_model,
}

_detailed_report = {
    'meta': _eval_meta,
    'metrics_readout_text': metrics_readout_text,
    'summary_table': summary_results,
    'classification_report': {'Base': _rep_base_dict, 'FT': _rep_ft_dict},
    'confusion_matrix': {'Base': _cm_base, 'FT': _cm_ft, 'labels': ['YES', 'NO']},
    'per_test_row': _per_rows,
    'plotting_hint': 'No PNGs written; use summary_table, recall_by_model, classification_report, per_test_row to chart.',
    'artifacts_written': [
        'eval_detailed_report.json',
        'eval_metrics_comparison_final.json',
        'eval_run_meta.json',
        'metrics_readout.txt',
        'per_row_generation_latency.csv',
    ],
}

with open(os.path.join(EVAL_DIR, 'eval_detailed_report.json'), 'w', encoding='utf-8') as f:
    json.dump(_detailed_report, f, indent=2, default=str)

with open(os.path.join(EVAL_DIR, 'eval_metrics_comparison_final.json'), 'w', encoding='utf-8') as f:
    json.dump(summary_results, f, indent=2, default=str)

with open(os.path.join(EVAL_DIR, 'metrics_readout.txt'), 'w', encoding='utf-8') as f:
    f.write(metrics_readout_text + '\n')

timing_df.round(6).to_csv(os.path.join(EVAL_DIR, 'per_row_generation_latency.csv'), index=False)

with open(os.path.join(EVAL_DIR, 'eval_run_meta.json'), 'w', encoding='utf-8') as f:
    json.dump(_eval_meta, f, indent=2, default=str)

print(metrics_readout_text)
print(f'\nSaved metrics-only bundle to: {EVAL_DIR}')


Eval output directory (reset): /content/drive/MyDrive/298B_WB2/evals/planner/v4
Eval on held-out test split: 300 row(s) (full test_df has 300; cap MAX_TEST_EVAL_ROWS=None).
Comparing Base vs. Fine-Tuned (k=6) on identical test prompts...


  0%|          | 0/300 [00:00<?, ?it/s]


--- Generation latency (wall-clock per k=6 generate call) ---
FT: mean=11.1485s  median=5.2400s  min=2.7769s  max=31.5831s  (n=300 calls, one per test row)
Base: mean=5.1659s  median=5.1577s  min=1.0691s  max=10.5633s  (n=300 calls, one per test row)
FT+Base per test row (both models): mean=16.3145s  median=11.6254s

Per-test-row latency (s) for one k=6 generation each (same row = same prompt):
 test_row  ft_gen_s  base_gen_s  both_models_s
        0    3.2066      6.6076         9.8141
        1    5.4294     10.2172        15.6466
        2    5.5589      6.0271        11.5860
        3   15.1845      2.3824        17.5669
        4    5.3431      2.9990         8.3421
        5    4.9036      8.8872        13.7907
        6    5.1827      4.3160         9.4987
        7    4.6591      4.3559         9.0150
        8   31.5831      7.1128        38.6959
        9    4.5856      7.9084        12.4940
       10   13.8494      2.7691        16.6185
       11    4.8471      3.7705      

G-Eval Base:   0%|          | 0/300 [00:00<?, ?it/s]

G-Eval row failed: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************HBIA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}
G-Eval row failed: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************HBIA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}
G-Eval row failed: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-************************************************************

G-Eval FT:   0%|          | 0/300 [00:00<?, ?it/s]

G-Eval row failed: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************HBIA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}
G-Eval row failed: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************HBIA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}
G-Eval row failed: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-************************************************************

/tmp/ipykernel_649/2141163571.py:342: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'created_utc': datetime.datetime.utcnow().replace(microsecond=0).isoformat() + 'Z',


In [13]:
# G-EVAL SUPPLEMENTAL RUN (With Schema, Parsing)
import os
import json
import random
import torch
import pandas as pd
import datetime
from tqdm.auto import tqdm
from openai import OpenAI
from google.colab import userdata

# --- 1. CONFIGURATION ---
client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))
EVAL_MODEL = "gpt-4o-mini"
NUM_SAMPLES = 50
DRIVE_V4_PATH = "/content/drive/MyDrive/298B_WB2/evals/planner/v4"
os.makedirs(DRIVE_V4_PATH, exist_ok=True)

# --- 2. THE RUBRIC ---
# Hyper-explicit about JSON keys
RUBRIC = """
You are a Senior Principal Engineer auditing an AI-generated Patch Plan for Apache Airflow.
Compare the 'Prediction' against the 'Ground Truth' based on the 'User Issue'.

Score 1-5 based on:
1. Logic & Intent: Does the prediction identify the same code-level root cause?
2. File Precision: Does it target the correct files?
3. Actionability: Is the plan grounded and technically sound?

Output your response in valid JSON format ONLY with these exact keys:
{
  "reasoning": "your technical justification",
  "score": <integer 1-5>
}
"""

# --- 3. INFERENCE HELPER ---
def get_prediction(prompt, use_adapter=True):
    inputs = TOKENIZER(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        if use_adapter:
            outputs = model.generate(**inputs, max_new_tokens=400, do_sample=False)
        else:
            with model.disable_adapter():
                outputs = model.generate(**inputs, max_new_tokens=400, do_sample=False)
    return TOKENIZER.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

# --- 4. EXECUTION LOOP ---
samples = test_df.sample(NUM_SAMPLES, random_state=42)
geval_records = []

print(f"Starting Corrected G-Eval on {NUM_SAMPLES} samples...")

for row in tqdm(samples.itertuples(), total=NUM_SAMPLES):
    user_issue_text, gt_text, sample_idx = row[1], row[3], row.Index

    ft_pred = get_prediction(user_issue_text, use_adapter=True)
    base_pred = get_prediction(user_issue_text, use_adapter=False)

    def audit_model(pred, label):
        prompt = f"{RUBRIC}\n\nISSUE: {user_issue_text}\nGT: {gt_text}\nPRED ({label}): {pred}"
        try:
            resp = client.chat.completions.create(
                model=EVAL_MODEL,
                messages=[{"role": "system", "content": "You are a senior code auditor. Respond only in JSON."},
                          {"role": "user", "content": prompt}],
                response_format={ "type": "json_object" }
            )
            res_data = json.loads(resp.choices[0].message.content)
            # Safe extraction logic handles case sensitivity
            return {
                "score": res_data.get("score") or res_data.get("Score") or 1,
                "reasoning": res_data.get("reasoning") or res_data.get("Reasoning") or "No justification provided."
            }
        except Exception as e:
            return {"score": 1, "reasoning": f"Audit Error: {e}"}

    ft_audit = audit_model(ft_pred, "Fine-Tuned")
    base_audit = audit_model(base_pred, "Base")

    geval_records.append({
        "sample_index": sample_idx,
        "ft_score": ft_audit['score'],
        "ft_reasoning": ft_audit['reasoning'],
        "base_score": base_audit['score'],
        "base_reasoning": base_audit['reasoning'],
        "ft_prediction": ft_pred,
        "base_prediction": base_pred,
        "gt": gt_text,
        "input": user_issue_text
    })

# --- 5. PERSISTENCE ---
avg_ft = sum(r['ft_score'] for r in geval_records) / len(geval_records)
avg_base = sum(r['base_score'] for r in geval_records) / len(geval_records)

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
filename = f"v4_geval_audit_FINAL_{timestamp}.json"
save_path = os.path.join(DRIVE_V4_PATH, filename)

with open(save_path, 'w') as f:
    json.dump({"summary": {"ft_avg": avg_ft, "base_avg": avg_base}, "details": geval_records}, f, indent=4)

print(f"\n--- G-EVAL REPAIRED ---\nFT Avg: {avg_ft:.2f}\nBase Avg: {avg_base:.2f}\nSaved to: {filename}")

Starting Corrected G-Eval on 50 samples...


  0%|          | 0/50 [00:00<?, ?it/s]


--- G-EVAL REPAIRED ---
FT Avg: 3.66
Base Avg: 3.74
Saved to: v4_geval_audit_FINAL_20260423_0641.json


In [6]:
# ============================================================
# CELL 6B — SAMPLE INFERENCE (The "Vibe Check")
# ============================================================
import random

# 1. Pick 2 random samples from the blind test set
sample_indices = random.sample(range(len(test_df)), 2)

print(f"--- STARTING VIBE CHECK ON {len(sample_indices)} TEST SAMPLES ---\n")

for i, idx in enumerate(sample_indices):
    row = test_df.iloc[idx]

    # Use the 'prompt' we built in Cell 3
    user_prompt = row['prompt']
    ground_truth = row['original_output']

    # Generate the prediction
    inputs = TOKENIZER(user_prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=300, do_sample=False)
    prediction = TOKENIZER.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    print(f"=== TEST SAMPLE #{i+1} (Index: {idx}) ===")
    print(f"\n[USER PROMPT/INPUT]:\n{user_prompt}")
    print("\n" + "-"*30)
    print(f"\n[GROUND TRUTH]:\n{ground_truth}")
    print("\n" + "-"*30)
    print(f"\n[AUTOBOT PREDICTION]:\n{prediction}")
    print("\n" + "="*60 + "\n")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


--- STARTING VIBE CHECK ON 2 TEST SAMPLES ---

=== TEST SAMPLE #1 (Index: 57) ===

[USER PROMPT/INPUT]:
<|im_start|>system
You are AutoBot Planner. Decide if a code change is required and output STRICT plain text only — no markdown, bold, or prose outside the required fields. Use exact fields: REQUIRES_CODE_CHANGE, REASON, and PLAN (for YES only). STRICT GROUNDING: You are an engineering tool. You must only select target files from the provided context blocks. Do not hallucinate paths or add file extensions not present in the retrieval. If a change is needed but no file is in context, identify the most relevant directory from the tree-sitter context. The user message may include --- RETRIEVAL EVIDENCE ---; treat it as context only. Do not output a CONFIDENCE field.<|im_end|>
<|im_start|>user
--- GRAPH RETRIEVAL CONTEXT ---
Historically similar tickets touched these candidate files:
providers/fab/src/airflow/providers/fab/auth_manager/api_fastapi/services/login.py
providers/fab/src/airf

In [12]:
# ============================================================
# CELL 7A — PUSH ADAPTER TO HUGGINGFACE HUB
# Since we are using Multi-LoRA on HF Inference endpoints,
# the adapter needs to be hosted on the Hub.
# ============================================================
from huggingface_hub import login
from google.colab import userdata
import os

# Get your WRITE token:
# 1. Go to huggingface.co and create an account (if you don't have one).
# 2. Navigate to Settings -> Access Tokens (or click here: https://huggingface.co/settings/tokens).
# 3. Create a new token and give it "Write" permissions (do not select just "Read").
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

REPO_NAME = 'cyborg1299/autobot-planner-lora-v4'

# Push it!
trainer.model.push_to_hub(REPO_NAME)
TOKENIZER.push_to_hub(REPO_NAME)
print(f"Successfully pushed pure LoRA adapter to https://huggingface.co/{REPO_NAME}")


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 39.6kB / 2.42GB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp7p8tqp9s/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Successfully pushed pure LoRA adapter to https://huggingface.co/cyborg1299/autobot-planner-lora-v4


In [ ]:
# ============================================================
# CELL 7B — PUSH ADAPTER TO HUGGINGFACE HUB (Post-Restart Version)
# ============================================================
from huggingface_hub import login
from google.colab import userdata
import os

# 1. Authenticate
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

# 2. Define Repo
REPO_NAME = 'cyborg1299/autobot-planner-lora-v3'

# 3. Push the 'model' variable directly (which holds your reloaded weights)
# We use save_embedding_layers=True to ensure compatibility with vLLM
model.push_to_hub(REPO_NAME)
TOKENIZER.push_to_hub(REPO_NAME)

print(f"Successfully pushed pure LoRA adapter to https://huggingface.co/{REPO_NAME}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   1%|1         | 3.41MB /  242MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpu32lqbr1/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Successfully pushed pure LoRA adapter to https://huggingface.co/cyborg1299/autobot-planner-lora-v2
